# 2 · Motion models

**Recap** ([theory](https://github.com/Rudip1/learn-state-estimation/blob/main/1_theory/02_motion_models.md)).
Planar poses compose with $a\oplus b$ and invert with $\ominus a$; their Jacobians $J_{1\oplus}, J_{2\oplus}$ carry
uncertainty through every motion. A differential-drive robot turns wheel travel $(d_l, d_r)$ into an arc
$s = (d_r+d_l)/2$, $\varphi = (d_r-d_l)/b$ and a robot-frame displacement $\delta(s,\varphi)$. The velocity model
compounds the arc of the commanded $(v,\omega)$; the odometry model decomposes the measured motion into
rotation–translation–rotation with motion-dependent noise. Dead reckoning integrates displacements,
$P_k = J_{1\oplus}P_{k-1}J_{1\oplus}^T + J_{2\oplus}QJ_{2\oplus}^T$, and its position uncertainty grows like
(distance)$^{3/2}$.

In [ ]:
# Installs the module when it is not available yet (e.g. on Colab). Building the C++ takes a few minutes.
try:
    import state_estimation
except ImportError:
    %pip install -q git+https://github.com/Rudip1/learn-state-estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import state_estimation as se
from state_estimation import plotting

plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6, 4.5)})

## Pose algebra

### ✏️ Exercise 1 — compounding by hand

Implement `compose_np(a, b)` from eq. (2.1) (remember to wrap the angle with `se.wrap_angle`), and check it
against the C++ version and against the product of homogeneous matrices.

In [ ]:
def compose_np(a, b):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
rng = np.random.default_rng(0)
for _ in range(100):
    a, b = rng.uniform(-3, 3, 3), rng.uniform(-3, 3, 3)
    assert np.allclose(compose_np(a, b), se.compose(a, b))
    assert np.allclose(se.to_matrix(compose_np(a, b)), se.to_matrix(a) @ se.to_matrix(b))
print("(2, 1, π/2) ⊕ (1, 0.5, π/2) =", compose_np([2, 1, np.pi / 2], [1, 0.5, np.pi / 2]).round(4))

### ✏️ Exercise 2 — the Jacobian with respect to the first pose

Differentiate (2.1) with respect to $a$ and implement `J1_np(a, b)`. Only the third column is interesting:
it is the lever arm of the heading.

In [ ]:
def J1_np(a, b):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
a, b = np.array([1.0, -2.0, 0.4]), np.array([0.7, 0.2, 1.1])
assert np.allclose(J1_np(a, b), se.compose_jacobian_1(a, b))
assert np.allclose(J1_np(a, b), se.numerical_jacobian(lambda x: se.compose(x, b), a), atol=1e-7)
print("✔ J1⊕ correct")

## Simulating a differential-drive robot

A robot with wheel radius 0.1 m, wheel base 0.5 m and 1024-tick encoders drives a circle and a figure eight.
`simulate_wheel_travel` adds encoder noise (variance proportional to the distance rolled) and quantises to whole
ticks; `run_dead_reckoning` integrates the readings and propagates the covariance (eq. 2.19–2.21).

In [ ]:
dd = se.DifferentialDrive(wheel_radius=0.1, wheel_base=0.5, ticks_per_revolution=1024)
dt, k = 0.1, 1e-4                       # sample time [s], encoder variance per metre [m]
x0, P0 = np.zeros(3), np.zeros((3, 3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, u) in zip(axes, [("circle", se.circle_controls(0.5, 2.0, 400)),
                                ("figure eight", se.figure_eight_controls(2.0, 250, dt))]):
    truth = se.integrate_controls(x0, u, dt)
    run = se.run_dead_reckoning(x0, P0, se.simulate_wheel_travel(u, dt, dd, k, se.Rng(1)), dd, k)
    plotting.new_axes(ax)
    plotting.plot_trajectory(ax, truth, "truth", "k")
    plotting.plot_trajectory(ax, run.poses, "dead reckoning", "C3")
    for i in range(0, len(u) + 1, 50):
        plotting.plot_ellipse(ax, run.poses[i], run.covariances[i], 0.95, color="C0")
    ax.set_title(f"{name}: final error {np.linalg.norm(run.poses[-1, :2] - truth[-1, :2]):.2f} m")
    ax.legend()
plt.show()

### ✏️ Exercise 3 — the drift exponent

Drive straight ($s = 0.05$ m per step, heading noise only, $\sigma = 0.005$ rad per step) for up to 400 steps
with `se.DeadReckoning.predict_displacement`, record $\sigma_y = \sqrt{P_{yy}}$ after each step, and fit the
exponent $p$ in $\sigma_y \propto n^p$ on a log–log scale (`np.polyfit`). Eq. (2.22) predicts $p \to 3/2$.

In [ ]:
def drift_exponent(n_steps=400, s=0.05, sigma=0.005):
    # ✏️ your code here
    raise NotImplementedError("exercise")


In [ ]:
p = drift_exponent()
print(f"fitted exponent {p:.3f}")
assert 1.45 < p < 1.55

## 🔨 Break it — when the heading uncertainty is large, the ellipse is the wrong shape

Repeat dead reckoning on the circle 2000 times and count how many final positions fall inside the predicted 95 %
ellipse (centred on the noise-free end point). With small encoder noise the count is 95 % and the mean squared
Mahalanobis distance is 2, as it must be for a 2-D Gaussian (chapter 5 calls this the NEES). With large noise the
heading uncertainty reaches ~0.4 rad, the cloud bends along the circle and the first-order covariance (2.20)
stops describing it.

In [ ]:
u = se.circle_controls(0.5, 2.0, 120)
ideal = se.DifferentialDrive(0.1, 0.5)
end_true = se.integrate_controls(x0, u, dt)[-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, kk in zip(axes, (1e-5, 3e-3)):
    rng = se.Rng(5)
    ends = []
    for _ in range(2000):
        run = se.run_dead_reckoning(x0, P0, se.simulate_wheel_travel(u, dt, ideal, kk, rng), ideal, kk)
        ends.append(run.poses[-1])
    ends = np.array(ends)
    pred = run.covariances[-1][:2, :2]
    d2 = np.array([se.mahalanobis_squared(e[:2], end_true[:2], pred) for e in ends])
    plotting.new_axes(ax)
    ax.plot(ends[:, 0], ends[:, 1], ".", ms=2, color="0.6")
    plotting.plot_ellipse(ax, end_true, pred, 0.95, color="C3", label="predicted 95 % (2.20)")
    ax.plot(*ends[:, :2].mean(0), "+", color="C0", ms=12, label="sample mean")
    ax.plot(*end_true[:2], "x", color="k", ms=9, label="noise-free end")
    ax.set_title(f"k = {kk}, σθ = {np.sqrt(run.covariances[-1][2, 2]):.2f} rad\n"
                 f"inside 95 % ellipse: {100 * np.mean(d2 <= 5.991):.1f} %, mean d² = {d2.mean():.2f} (ideal 2)")
    ax.legend(fontsize=8)
plt.show()

## 🔨 Break it — a systematic error is not noise

The robot's real wheel base is 2 % larger than the value used for dead reckoning. Every run now curves the same
way: the cloud of final positions moves away from the truth, and the predicted ellipse no longer contains it.
No amount of process noise tuning fixes this; calibration does.

In [ ]:
true_dd, model_dd = se.DifferentialDrive(0.1, 0.51), se.DifferentialDrive(0.1, 0.50)
u = se.figure_eight_controls(2.0, 250, dt)
truth = se.integrate_controls(x0, u, dt)
rng = se.Rng(9)
ends = []
for _ in range(300):
    w = se.simulate_wheel_travel(u, dt, true_dd, 1e-5, rng)      # the real robot
    run = se.run_dead_reckoning(x0, P0, w, model_dd, 1e-5)        # our (wrong) model of it
    ends.append(run.poses[-1])
ends = np.array(ends)
d2 = se.mahalanobis_squared(truth[-1, :2], ends[:, :2].mean(0), run.covariances[-1][:2, :2])
fig, (ax, zoom) = plt.subplots(1, 2, figsize=(12, 5))
for a in (ax, zoom):
    plotting.new_axes(a)
    plotting.plot_trajectory(a, truth, "truth", "k")
    plotting.plot_trajectory(a, run.poses, "dead reckoning, 2 % wheel-base error", "C3")
    a.plot(ends[:, 0], ends[:, 1], ".", ms=3, color="0.5", label="final pose of 300 runs")
    plotting.plot_ellipse(a, run.poses[-1], run.covariances[-1], 0.95, color="C0", label="predicted 95 %")
    a.plot(*truth[-1, :2], "kx", ms=10)
c = ends[:, :2].mean(0)
zoom.set_xlim(c[0] - 0.6, c[0] + 0.6); zoom.set_ylim(c[1] - 0.6, c[1] + 0.6)
zoom.set_title("zoom on the end point (× = true end)")
zoom.legend(fontsize=8)
plt.show()
print(f"squared Mahalanobis distance of the true end point: {d2:.1f} (95 % gate {se.chi2_quantile(0.95, 2):.2f})")

## What to remember

- Every planar motion is a compounding $x_k = x_{k-1}\oplus\delta$; its Jacobians $J_{1\oplus}, J_{2\oplus}$ are all
  you need to propagate uncertainty.
- Encoder odometry gives $(s, \varphi)$ from wheel travel; prefer it to integrating commanded velocities.
- The odometry model perturbs rotation–translation–rotation; its samples form a banana.
- Dead reckoning drifts as (distance)$^{3/2}$ — only external measurements stop it.
- Systematic errors (calibration) are not covered by any covariance.